# Byte-Pair Encoding (BPE)

**难度:** Hard

实现字节对编码（BPE）分词。

BPE 通过迭代合并最频繁的相邻符号对来构建子词词表，广泛用于现代语言模型。

**签名:** `SimpleBPE()`（类）

**方法:**
- `train(corpus: list[str], num_merges: int)` — 从词列表中学习合并规则
- `encode(text: str) -> list[str]` — 将文本分词为子词 token

**约束:**
- 使用 `</w>` 标记词边界
- `self.merges` 按顺序存储学习到的合并对
- `encode` 按顺序应用合并规则

**提示:**
train：每个词拆分为字符 + `</w>`，统计相邻对频率，合并最高频对，重复
encode：按学习到的合并顺序将文本拆分为子词 token

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ INTERVIEW

# ---- 手写 BPE ----
# 面试要点：BPE 不涉及 PyTorch，纯 Python 实现
# 核心考点：训练时的贪心合并 + 编码时的规则应用

class SimpleBPE:
    def __init__(self):
        self.merges = []

    def train(self, corpus, num_merges):
        # 初始化：每个词 → 字符元组 + '</w>'
        # 面试考点：为什么用 tuple 而不是 list？
        # 答：tuple 可以做字典 key（hashable），list 不行
        vocab = {}
        for word in corpus:
            symbols = tuple(word) + ('</w>',)
            vocab[symbols] = vocab.get(symbols, 0) + 1

        self.merges = []
        for _ in range(num_merges):
            # 统计 pair 频率
            pairs = {}
            for word, freq in vocab.items():
                for i in range(len(word) - 1):
                    pair = (word[i], word[i + 1])
                    pairs[pair] = pairs.get(pair, 0) + freq

            if not pairs:
                break

            # 选最高频 pair
            best = max(pairs, key=pairs.get)
            self.merges.append(best)

            # 执行合并
            new_vocab = {}
            for word, freq in vocab.items():
                new_word = []
                i = 0
                while i < len(word):
                    if i < len(word) - 1 and (word[i], word[i + 1]) == best:
                        new_word.append(word[i] + word[i + 1])
                        i += 2
                    else:
                        new_word.append(word[i])
                        i += 1
                new_vocab[tuple(new_word)] = freq
            vocab = new_vocab

    def encode(self, text):
        all_tokens = []
        for word in text.split():
            symbols = list(word) + ['</w>']
            # 面试考点：必须按训练顺序应用合并！
            # 顺序决定了优先级：先学的合并更重要
            for a, b in self.merges:
                i = 0
                while i < len(symbols) - 1:
                    if symbols[i] == a and symbols[i + 1] == b:
                        symbols = symbols[:i] + [a + b] + symbols[i + 2:]
                    else:
                        i += 1
            all_tokens.extend(symbols)
        return all_tokens

In [ ]:
# Demo
bpe = SimpleBPE()
bpe.train(['low', 'low', 'low', 'lower', 'newest', 'widest'], num_merges=10)
print('Merges:', bpe.merges)
print('Encode:', bpe.encode('low lower newest'))

In [ ]:
from torch_judge import check
check('bpe')

## 📝 核心思路总结

1. **BPE 的核心：贪心合并最高频相邻对**：从字符级开始，迭代合并最频繁的 pair
2. **`</w>` 标记词边界**：区分词尾字符和独立字符（如 "at" vs "at</w>"）
3. **encode 按训练顺序应用合并规则**：顺序很重要，先学的合并优先级更高